In [ ]:
import kagglehub

mlpath = kagglehub.dataset_download("sherinclaudia/movielens")

print("Path to dataset files:", mlpath)

krpath = kagglehub.dataset_download("arashnic/kuairec-recommendation-system-data-density-100")

print("Path to dataset files:", krpath)

Using Colab cache for faster access to the 'movielens' dataset.
Path to dataset files: /kaggle/input/movielens
Using Colab cache for faster access to the 'kuairec-recommendation-system-data-density-100' dataset.
Path to dataset files: /kaggle/input/kuairec-recommendation-system-data-density-100


In [ ]:
import pandas as pd
kr_bigm=pd.read_csv("/kaggle/input/kuairec-recommendation-system-data-density-100/KuaiRec 2.0/data/big_matrix.csv")
print(kr_bigm.head(5))

   user_id  video_id  play_duration  video_duration                     time  \
0        0      3649          13838           10867  2020-07-05 00:08:23.438   
1        0      9598          13665           10984  2020-07-05 00:13:41.297   
2        0      5262            851            7908  2020-07-05 00:16:06.687   
3        0      1963            862            9590  2020-07-05 00:20:26.792   
4        0      8234            858           11000  2020-07-05 00:43:05.128   

       date     timestamp  watch_ratio  
0  20200705  1.593879e+09     1.273397  
1  20200705  1.593879e+09     1.244082  
2  20200705  1.593879e+09     0.107613  
3  20200705  1.593880e+09     0.089885  
4  20200705  1.593881e+09     0.078000  


In [ ]:
kr_smallm=pd.read_csv("/kaggle/input/kuairec-recommendation-system-data-density-100/KuaiRec 2.0/data/small_matrix.csv")
print(kr_smallm.head(5))

   user_id  video_id  play_duration  video_duration                     time  \
0       14       148           4381            6067  2020-07-05 05:27:48.378   
1       14       183          11635            6100  2020-07-05 05:28:00.057   
2       14      3649          22422           10867  2020-07-05 05:29:09.479   
3       14      5262           4479            7908  2020-07-05 05:30:43.285   
4       14      8234           4602           11000  2020-07-05 05:35:43.459   

         date     timestamp  watch_ratio  
0  20200705.0  1.593898e+09     0.722103  
1  20200705.0  1.593898e+09     1.907377  
2  20200705.0  1.593898e+09     2.063311  
3  20200705.0  1.593898e+09     0.566388  
4  20200705.0  1.593899e+09     0.418364  


In [ ]:
kr_cats=pd.read_csv("/kaggle/input/kuairec-recommendation-system-data-density-100/KuaiRec 2.0/data/item_categories.csv")
print(kr_cats.head(5))

   video_id     feat
0         0      [8]
1         1  [27, 9]
2         2      [9]
3         3     [26]
4         4      [5]


In [ ]:
ml_movies=pd.read_csv("/kaggle/input/movielens/movies.dat", sep='::', names=['MovieID', 'Title', 'Genres'], engine='python', encoding='latin-1')
# Fill NaN values with empty strings before splitting, then filter out empty strings from lists
ml_movies['Genres'] = ml_movies['Genres'].fillna('').str.split('|').apply(lambda x: [g for g in x if g])
print(ml_movies.head(5))

   MovieID                               Title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                             Genres  
0   [Animation, Children's, Comedy]  
1  [Adventure, Children's, Fantasy]  
2                 [Comedy, Romance]  
3                   [Comedy, Drama]  
4                          [Comedy]  


In [ ]:
ml_ratings=pd.read_csv("/kaggle/input/movielens/ratings.dat", sep='::', names=["UID", "MovieID", "Rating", "Timestamp"], engine='python', encoding='latin-1')
print(ml_ratings.head(5))

   UID  MovieID  Rating  Timestamp
0    1     1193       5  978300760
1    1      661       3  978302109
2    1      914       3  978301968
3    1     3408       4  978300275
4    1     2355       5  978824291


In [ ]:
ml_users=pd.read_csv("/kaggle/input/movielens/users.dat", sep='::', names=["UID", "SEX", "AGE", "OCC", "PIN"], engine='python', encoding='latin-1')
print(ml_users.head(5))

   UID SEX  AGE  OCC    PIN
0    1   F    1   10  48067
1    2   M   56   16  70072
2    3   M   25   15  55117
3    4   M   45    7  02460
4    5   M   25   20  55455


In [ ]:
assert set(kr_smallm.user_id).issubset(set(kr_bigm.user_id))
assert set(kr_smallm.video_id).issubset(set(kr_bigm.video_id))

In [ ]:
print(ml_movies.shape)
print(ml_ratings.shape)
print(ml_users.shape)

(3883, 3)
(1000209, 4)
(6040, 5)


In [ ]:
threshold = 2.0
kr_bigm['label'] = (kr_bigm['watch_ratio'] > threshold).astype(int)
kr_smallm['label'] = (kr_smallm['watch_ratio'] > threshold).astype(int)
print(kr_bigm['label'].mean(), kr_smallm['label'].mean())

0.07472703671256263 0.04643894991414648


In [ ]:
import ast
import numpy as np
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer

mlb_kr = MultiLabelBinarizer()
kr_cat_matrix = mlb_kr.fit_transform(kr_cats['feat'])
kr_cat_df = pd.DataFrame(kr_cat_matrix, columns=[f"cat_{c}" for c in mlb_kr.classes_])
kr_cat_df['video_id'] = kr_cats['video_id'].values

n_kr_cats = kr_cat_df.shape[1] - 1
print(f"KuaiRec: {n_kr_cats} unique category tags, {len(kr_cat_df)} items")

KR_THRESHOLD = 2.0

kr_bigm = kr_bigm[['user_id', 'video_id', 'watch_ratio']].copy()
kr_smallm = kr_smallm[['user_id', 'video_id', 'watch_ratio']].copy()

kr_bigm['label'] = (kr_bigm['watch_ratio'] > KR_THRESHOLD).astype(int)
kr_smallm['label'] = (kr_smallm['watch_ratio'] > KR_THRESHOLD).astype(int)

print(f"KuaiRec big_matrix positive rate: {kr_bigm['label'].mean():.4f}")
print(f"KuaiRec small_matrix positive rate: {kr_smallm['label'].mean():.4f}")

missing_users = set(kr_smallm.user_id) - set(kr_bigm.user_id)
missing_items = set(kr_smallm.video_id) - set(kr_bigm.video_id)
assert not missing_users, f"{len(missing_users)} eval users not seen in training!"
assert not missing_items, f"{len(missing_items)} eval items not seen in training!"
print("KuaiRec: small_matrix fully covered by big_matrix vocabulary OK")

kr_user2idx = {u: i for i, u in enumerate(sorted(kr_bigm.user_id.unique()))}
kr_item2idx = {v: i for i, v in enumerate(sorted(kr_bigm.video_id.unique()))}

for df in (kr_bigm, kr_smallm):
    df['user_idx'] = df['user_id'].map(kr_user2idx)
    df['item_idx'] = df['video_id'].map(kr_item2idx)

kr_cat_df['item_idx'] = kr_cat_df['video_id'].map(kr_item2idx)
kr_cat_df = kr_cat_df.dropna(subset=['item_idx'])  # drop items not in train vocab
kr_cat_df['item_idx'] = kr_cat_df['item_idx'].astype(int)

n_kr_items = len(kr_item2idx)
kr_feat_cols = [c for c in kr_cat_df.columns if c.startswith('cat_')]
kr_item_feat_matrix = np.zeros((n_kr_items, len(kr_feat_cols)), dtype=np.float32)
kr_item_feat_matrix[kr_cat_df['item_idx'].values] = kr_cat_df[kr_feat_cols].values

print(f"KuaiRec: {len(kr_user2idx)} users, {n_kr_items} items, feature dim {kr_item_feat_matrix.shape[1]}")

mlb_ml = MultiLabelBinarizer()
ml_genre_matrix = mlb_ml.fit_transform(ml_movies['Genres'])
ml_genre_df = pd.DataFrame(ml_genre_matrix, columns=[f"genre_{g}" for g in mlb_ml.classes_])
ml_genre_df['MovieID'] = ml_movies['MovieID'].values

n_ml_genres = ml_genre_df.shape[1] - 1
print(f"MovieLens: {n_ml_genres} unique genres, {len(ml_genre_df)} movies")

ML_THRESHOLD = 3.5  # ratings are 1-5; >3.5 treated as implicit positive

ml_ratings = ml_ratings[['UID', 'MovieID', 'Rating']].copy()
ml_ratings['label'] = (ml_ratings['Rating'] > ML_THRESHOLD).astype(int)

print(f"MovieLens positive rate: {ml_ratings['label'].mean():.4f}")

ml_user2idx = {u: i for i, u in enumerate(sorted(ml_ratings.UID.unique()))}
ml_item2idx = {m: i for i, m in enumerate(sorted(ml_ratings.MovieID.unique()))}

ml_ratings['user_idx'] = ml_ratings['UID'].map(ml_user2idx)
ml_ratings['item_idx'] = ml_ratings['MovieID'].map(ml_item2idx)

ml_genre_df['item_idx'] = ml_genre_df['MovieID'].map(ml_item2idx)
ml_genre_df = ml_genre_df.dropna(subset=['item_idx'])
ml_genre_df['item_idx'] = ml_genre_df['item_idx'].astype(int)

n_ml_items = len(ml_item2idx)
ml_feat_cols = [c for c in ml_genre_df.columns if c.startswith('genre_')]
ml_item_feat_matrix = np.zeros((n_ml_items, len(ml_feat_cols)), dtype=np.float32)
ml_item_feat_matrix[ml_genre_df['item_idx'].values] = ml_genre_df[ml_feat_cols].values

print(f"MovieLens: {len(ml_user2idx)} users, {n_ml_items} items, feature dim {ml_item_feat_matrix.shape[1]}")

from sklearn.model_selection import train_test_split

ml_train, ml_test = train_test_split(
    ml_ratings, test_size=0.2, random_state=42, stratify=ml_ratings['label']
)
print(f"MovieLens: {len(ml_train)} train / {len(ml_test)} test interactions")


print("\n--- Common schema ready ---")
print("KuaiRec train:", kr_bigm[['user_idx','item_idx','label']].shape,
      "| feature matrix:", kr_item_feat_matrix.shape)
print("KuaiRec eval :", kr_smallm[['user_idx','item_idx','label']].shape)
print("MovieLens train:", ml_train[['user_idx','item_idx','label']].shape,
      "| feature matrix:", ml_item_feat_matrix.shape)
print("MovieLens test :", ml_test[['user_idx','item_idx','label']].shape)

KuaiRec: 14 unique category tags, 10728 items
KuaiRec big_matrix positive rate: 0.0747
KuaiRec small_matrix positive rate: 0.0464
KuaiRec: small_matrix fully covered by big_matrix vocabulary OK
KuaiRec: 7176 users, 10728 items, feature dim 14
MovieLens: 18 unique genres, 3883 movies
MovieLens positive rate: 0.5752
MovieLens: 6040 users, 3706 items, feature dim 18
MovieLens: 800167 train / 200042 test interactions

--- Common schema ready ---
KuaiRec train: (12530806, 3) | feature matrix: (10728, 14)
KuaiRec eval : (4676570, 3)
MovieLens train: (800167, 3) | feature matrix: (3706, 18)
MovieLens test : (200042, 3)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# =========================================================
# DeepFM with toggleable content features
# =========================================================
class DeepFM(nn.Module):
    def __init__(self, n_users, n_items, n_content_feats, emb_dim=64,
                 use_content=True, deep_layers=(128, 64)):
        super().__init__()
        self.use_content = use_content
        self.emb_dim = emb_dim

        # ID embeddings (always present)
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.item_emb = nn.Embedding(n_items, emb_dim)

        # FM first-order (linear) terms
        self.user_bias = nn.Embedding(n_users, 1)
        self.item_bias = nn.Embedding(n_items, 1)
        self.global_bias = nn.Parameter(torch.zeros(1))

        # Content projection into the SAME embedding space, so it can
        # participate in FM second-order interactions with user/item embeddings
        if use_content:
            self.content_proj = nn.Linear(n_content_feats, emb_dim, bias=False)
            self.content_bias_proj = nn.Linear(n_content_feats, 1, bias=False)

        # Deep component input size depends on whether content is concatenated
        deep_input_dim = emb_dim * 2 + (emb_dim if use_content else 0)
        layers = []
        prev = deep_input_dim
        for h in deep_layers:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(0.2)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.deep = nn.Sequential(*layers)

        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, user_idx, item_idx, content_feat=None):
        u_emb = self.user_emb(user_idx)          # (B, d)
        i_emb = self.item_emb(item_idx)          # (B, d)

        vectors = [u_emb, i_emb]
        linear_terms = self.global_bias + self.user_bias(user_idx).squeeze(-1) \
                       + self.item_bias(item_idx).squeeze(-1)

        if self.use_content:
            c_emb = self.content_proj(content_feat)          # (B, d)
            c_lin = self.content_bias_proj(content_feat).squeeze(-1)
            vectors.append(c_emb)
            linear_terms = linear_terms + c_lin

        # FM second-order term: 0.5 * [(sum v)^2 - sum(v^2)] summed over dims
        stacked = torch.stack(vectors, dim=1)         # (B, n_fields, d)
        sum_sq = stacked.sum(dim=1).pow(2)
        sq_sum = stacked.pow(2).sum(dim=1)
        fm_second_order = 0.5 * (sum_sq - sq_sum).sum(dim=1)

        # Deep component
        deep_input = torch.cat(vectors, dim=-1)
        deep_out = self.deep(deep_input).squeeze(-1)

        logit = linear_terms + fm_second_order + deep_out
        return logit  # apply sigmoid outside (use BCEWithLogitsLoss)


# =========================================================
# Wide & Deep with toggleable content features
# =========================================================
class WideAndDeep(nn.Module):
    def __init__(self, n_users, n_items, n_content_feats, emb_dim=64,
                 use_content=True, deep_layers=(128, 64)):
        super().__init__()
        self.use_content = use_content

        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.item_emb = nn.Embedding(n_items, emb_dim)

        # Wide component: user bias + item bias + (optional) content linear term
        self.user_bias = nn.Embedding(n_users, 1)
        self.item_bias = nn.Embedding(n_items, 1)
        self.global_bias = nn.Parameter(torch.zeros(1))
        if use_content:
            self.wide_content = nn.Linear(n_content_feats, 1, bias=False)

        # Deep component
        deep_input_dim = emb_dim * 2 + (n_content_feats if use_content else 0)
        layers = []
        prev = deep_input_dim
        for h in deep_layers:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(0.2)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.deep = nn.Sequential(*layers)

        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, user_idx, item_idx, content_feat=None):
        u_emb = self.user_emb(user_idx)
        i_emb = self.item_emb(item_idx)

        wide_out = self.global_bias + self.user_bias(user_idx).squeeze(-1) \
                   + self.item_bias(item_idx).squeeze(-1)

        deep_parts = [u_emb, i_emb]
        if self.use_content:
            wide_out = wide_out + self.wide_content(content_feat).squeeze(-1)
            deep_parts.append(content_feat)  # raw content vector into deep tower

        deep_input = torch.cat(deep_parts, dim=-1)
        deep_out = self.deep(deep_input).squeeze(-1)

        return wide_out + deep_out


# =========================================================
# SVD baseline (Koren-style: global bias + user/item bias + latent factors)
# =========================================================
class SVDBaseline(nn.Module):
    def __init__(self, n_users, n_items, emb_dim=64):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.item_emb = nn.Embedding(n_items, emb_dim)
        self.user_bias = nn.Embedding(n_users, 1)
        self.item_bias = nn.Embedding(n_items, 1)
        self.global_bias = nn.Parameter(torch.zeros(1))
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, user_idx, item_idx, content_feat=None):  # content_feat ignored
        dot = (self.user_emb(user_idx) * self.item_emb(item_idx)).sum(dim=-1)
        bias = self.global_bias + self.user_bias(user_idx).squeeze(-1) \
               + self.item_bias(item_idx).squeeze(-1)
        return dot + bias


# =========================================================
# LightGCN (no content features, propagation over interaction graph)
# =========================================================
class LightGCN(nn.Module):
    def __init__(self, n_users, n_items, emb_dim=64, n_layers=3):
        super().__init__()
        self.n_users = n_users
        self.n_items = n_items
        self.n_layers = n_layers
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.item_emb = nn.Embedding(n_items, emb_dim)
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)
        self.norm_adj = None  # set via set_graph()

    def set_graph(self, norm_adj_sparse_tensor):
        """norm_adj: symmetrically-normalized (n_users+n_items) x (n_users+n_items)
        sparse adjacency, precomputed once from the training interactions."""
        self.norm_adj = norm_adj_sparse_tensor

    def propagate(self):
        all_emb = torch.cat([self.user_emb.weight, self.item_emb.weight], dim=0)
        embs = [all_emb]
        for _ in range(self.n_layers):
            all_emb = torch.sparse.mm(self.norm_adj, all_emb)
            embs.append(all_emb)
        final_emb = torch.stack(embs, dim=0).mean(dim=0)  # layer-combination
        final_user, final_item = final_emb[:self.n_users], final_emb[self.n_users:]
        return final_user, final_item

    def forward(self, user_idx, item_idx, content_feat=None):
        final_user, final_item = self.propagate()
        u = final_user[user_idx]
        i = final_item[item_idx]
        return (u * i).sum(dim=-1)

In [ ]:
import scipy.sparse as sp
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import time

# =========================================================
# 1. Build normalized adjacency matrix for LightGCN
# =========================================================
def build_norm_adj(user_idx_arr, item_idx_arr, n_users, n_items):
    """
    Builds the symmetrically normalized bipartite adjacency matrix
    D^-1/2 (A) D^-1/2 used by LightGCN, as a torch sparse tensor.
    """
    n_nodes = n_users + n_items

    # User-item interactions -> shift item indices by n_users
    rows = np.concatenate([user_idx_arr, item_idx_arr + n_users])
    cols = np.concatenate([item_idx_arr + n_users, user_idx_arr])
    vals = np.ones(len(rows), dtype=np.float32)

    adj = sp.coo_matrix((vals, (rows, cols)), shape=(n_nodes, n_nodes))
    adj = adj.tocsr()
    adj.data[:] = 1.0  # dedupe multi-interactions to binary edges
    adj = adj.tocoo()

    # Degree normalization: D^-1/2 A D^-1/2
    deg = np.array(adj.sum(axis=1)).flatten()
    deg[deg == 0] = 1e-8
    d_inv_sqrt = np.power(deg, -0.5)
    d_mat = sp.diags(d_inv_sqrt)
    norm_adj = d_mat @ adj @ d_mat
    norm_adj = norm_adj.tocoo()

    # Convert to torch sparse tensor
    indices = torch.LongTensor(np.vstack([norm_adj.row, norm_adj.col]))
    values = torch.FloatTensor(norm_adj.data)
    shape = torch.Size(norm_adj.shape)
    return torch.sparse_coo_tensor(indices, values, shape).coalesce()


# =========================================================
# 2. Dataset with negative sampling for implicit feedback
# =========================================================
class InteractionDataset(Dataset):
    """
    Wraps positive interactions and does negative sampling on the fly.
    Only positive (label==1) rows are used as anchors; for each we sample
    a random item the user hasn't interacted with as the negative.
    """
    def __init__(self, df, n_items, user_pos_items, n_neg=1):
        self.pos_df = df[df['label'] == 1].reset_index(drop=True)
        self.n_items = n_items
        self.user_pos_items = user_pos_items  # dict: user_idx -> set(item_idx)
        self.n_neg = n_neg

    def __len__(self):
        return len(self.pos_df)

    def _sample_neg(self, user_idx):
        pos_set = self.user_pos_items.get(user_idx, set())
        while True:
            neg = np.random.randint(0, self.n_items)
            if neg not in pos_set:
                return neg

    def __getitem__(self, idx):
        row = self.pos_df.iloc[idx]
        user_idx = int(row['user_idx'])
        pos_item = int(row['item_idx'])
        neg_item = self._sample_neg(user_idx)
        return user_idx, pos_item, neg_item


def build_user_pos_items(df):
    pos = df[df['label'] == 1]
    return pos.groupby('user_idx')['item_idx'].apply(set).to_dict()


# =========================================================
# 3. Training loop (shared across all 4 models)
# =========================================================
def train_model(model, train_df, n_items, item_feat_matrix=None,
                 use_content=False, n_epochs=15, batch_size=2048,
                 lr=1e-3, weight_decay=1e-6, device='cuda',
                 norm_adj=None, verbose=True):
    """
    Trains any of DeepFM / WideAndDeep / SVDBaseline / LightGCN using
    BPR-style pairwise loss on positive vs. sampled-negative items.

    item_feat_matrix: (n_items, n_feat) numpy array. Required if use_content=True.
    norm_adj: required for LightGCN, ignored otherwise (set via model.set_graph).
    """
    model = model.to(device)
    if norm_adj is not None:
        model.set_graph(norm_adj.to(device))

    if item_feat_matrix is not None:
        item_feat_tensor = torch.tensor(item_feat_matrix, dtype=torch.float32).to(device)
    else:
        item_feat_tensor = None

    user_pos_items = build_user_pos_items(train_df)
    dataset = InteractionDataset(train_df, n_items, user_pos_items)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=2)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    for epoch in range(n_epochs):
        model.train()
        total_loss = 0.0
        t0 = time.time()

        for user_idx, pos_item, neg_item in loader:
            user_idx = user_idx.to(device)
            pos_item = pos_item.to(device)
            neg_item = neg_item.to(device)

            if use_content:
                pos_feat = item_feat_tensor[pos_item]
                neg_feat = item_feat_tensor[neg_item]
            else:
                pos_feat = None
                neg_feat = None

            pos_score = model(user_idx, pos_item, pos_feat)
            neg_score = model(user_idx, neg_item, neg_feat)

            # BPR loss: encourage pos_score > neg_score
            loss = -torch.log(torch.sigmoid(pos_score - neg_score) + 1e-8).mean()

            # small L2 reg on embeddings only (standard BPR regularization)
            reg = 0.0
            if hasattr(model, 'user_emb'):
                reg += model.user_emb.weight.norm(2).pow(2) / user_idx.shape[0]
            loss = loss + weight_decay * reg

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        if verbose:
            print(f"Epoch {epoch+1}/{n_epochs} | loss={total_loss/len(loader):.4f} | "
                  f"time={time.time()-t0:.1f}s")

    return model

In [ ]:
import numpy as np
import torch
import pandas as pd

# =========================================================
# 1. Ranking evaluation: Recall@K and NDCG@K
# =========================================================
@torch.no_grad()
def evaluate_model(model, eval_df, n_items, item_feat_matrix=None,
                    use_content=False, k_list=(10, 20), device='cuda',
                    train_user_pos_items=None, batch_users=512):
    """
    Full-ranking evaluation: for each user in eval_df with at least one
    positive, rank ALL items, mask out items the user saw in TRAINING
    (standard practice — don't re-recommend training positives), then
    score Recall@K / NDCG@K against the eval positives.
    """
    model.eval()

    eval_pos = eval_df[eval_df['label'] == 1]
    eval_user_pos = eval_pos.groupby('user_idx')['item_idx'].apply(set).to_dict()
    eval_users = sorted(eval_user_pos.keys())

    if item_feat_matrix is not None:
        item_feat_tensor = torch.tensor(item_feat_matrix, dtype=torch.float32).to(device)
    else:
        item_feat_tensor = None

    all_items = torch.arange(n_items).to(device)

    recalls = {k: [] for k in k_list}
    ndcgs = {k: [] for k in k_list}

    for start in range(0, len(eval_users), batch_users):
        batch_user_ids = eval_users[start:start + batch_users]
        user_tensor = torch.tensor(batch_user_ids, dtype=torch.long).to(device)

        # Score every item for every user in this batch
        # Expand: (batch_users, n_items)
        scores = torch.zeros(len(batch_user_ids), n_items, device=device)
        for i, u in enumerate(batch_user_ids):
            u_rep = torch.full((n_items,), u, dtype=torch.long, device=device)
            if use_content:
                s = model(u_rep, all_items, item_feat_tensor)
            else:
                s = model(u_rep, all_items, None)
            scores[i] = s

        # Mask out items seen during training (don't recommend what's already seen)
        if train_user_pos_items is not None:
            for i, u in enumerate(batch_user_ids):
                seen = train_user_pos_items.get(u, set())
                if seen:
                    idx = torch.tensor(list(seen), dtype=torch.long, device=device)
                    scores[i, idx] = -1e9

        # Top-K ranking
        max_k = max(k_list)
        topk_scores, topk_idx = torch.topk(scores, max_k, dim=1)
        topk_idx = topk_idx.cpu().numpy()

        for i, u in enumerate(batch_user_ids):
            pos_items = eval_user_pos[u]
            ranked = topk_idx[i]

            for k in k_list:
                ranked_k = ranked[:k]
                hits = np.isin(ranked_k, list(pos_items))

                # Recall@K
                recall = hits.sum() / min(len(pos_items), k) if len(pos_items) > 0 else 0.0
                recalls[k].append(recall)

                # NDCG@K
                dcg = np.sum(hits / np.log2(np.arange(2, k + 2)))
                ideal_hits = min(len(pos_items), k)
                idcg = np.sum(1.0 / np.log2(np.arange(2, ideal_hits + 2))) if ideal_hits > 0 else 0.0
                ndcg = dcg / idcg if idcg > 0 else 0.0
                ndcgs[k].append(ndcg)

    results = {}
    for k in k_list:
        results[f'Recall@{k}'] = np.mean(recalls[k])
        results[f'NDCG@{k}'] = np.mean(ndcgs[k])
    return results


# =========================================================
# 2. Config: model builders per dataset
# =========================================================
def build_models(n_users, n_items, n_content_feats, emb_dim=64):
    return {
        'DeepFM_on':      DeepFM(n_users, n_items, n_content_feats, emb_dim, use_content=True),
        'DeepFM_off':     DeepFM(n_users, n_items, n_content_feats, emb_dim, use_content=False),
        'WideDeep_on':    WideAndDeep(n_users, n_items, n_content_feats, emb_dim, use_content=True),
        'WideDeep_off':   WideAndDeep(n_users, n_items, n_content_feats, emb_dim, use_content=False),
        'SVD':            SVDBaseline(n_users, n_items, emb_dim),
        'LightGCN':       LightGCN(n_users, n_items, emb_dim, n_layers=3),
    }


# =========================================================
# 3. Run all 12 experiments (6 models x 2 datasets)
# =========================================================
def run_all_experiments(train_df, eval_df, n_users, n_items,
                         item_feat_matrix, dataset_name,
                         n_epochs=15, seeds=(0, 1, 2), device='cuda'):

    n_content_feats = item_feat_matrix.shape[1]
    norm_adj = build_norm_adj(
        train_df['user_idx'].values, train_df['item_idx'].values, n_users, n_items
    )
    train_user_pos_items = build_user_pos_items(train_df)

    all_results = []

    for seed in seeds:
        torch.manual_seed(seed)
        np.random.seed(seed)

        models = build_models(n_users, n_items, n_content_feats)

        for name, model in models.items():
            use_content = name.endswith('_on')  # DeepFM_on / WideDeep_on
            is_lightgcn = (name == 'LightGCN')

            print(f"\n=== [{dataset_name}] Training {name} (seed={seed}) ===")

            trained = train_model(
                model, train_df, n_items,
                item_feat_matrix=item_feat_matrix if use_content else None,
                use_content=use_content,
                n_epochs=n_epochs,
                device=device,
                norm_adj=norm_adj if is_lightgcn else None,
            )

            metrics = evaluate_model(
                trained, eval_df, n_items,
                item_feat_matrix=item_feat_matrix if use_content else None,
                use_content=use_content,
                train_user_pos_items=train_user_pos_items,
                device=device,
            )

            row = {'dataset': dataset_name, 'model': name, 'seed': seed, **metrics}
            all_results.append(row)
            print(row)

    return pd.DataFrame(all_results)


# =========================================================
# 4. Actually run it for both datasets
# =========================================================
device = 'cuda' if torch.cuda.is_available() else 'cpu'

kr_results = run_all_experiments(
    train_df=kr_bigm, eval_df=kr_smallm,
    n_users=len(kr_user2idx), n_items=n_kr_items,
    item_feat_matrix=kr_item_feat_matrix,
    dataset_name='KuaiRec', device=device,
)

ml_results = run_all_experiments(
    train_df=ml_train, eval_df=ml_test,
    n_users=len(ml_user2idx), n_items=n_ml_items,
    item_feat_matrix=ml_item_feat_matrix,
    dataset_name='MovieLens', device=device,
)

all_results = pd.concat([kr_results, ml_results], ignore_index=True)
all_results.to_csv('all_results.csv', index=False)
print(all_results.groupby(['dataset', 'model']).mean(numeric_only=True))

/tmp/ipykernel_928/3213219399.py:40: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  return torch.sparse_coo_tensor(indices, values, shape).coalesce()



=== [KuaiRec] Training DeepFM_on (seed=0) ===
Epoch 1/15 | loss=0.3063 | time=97.7s
Epoch 2/15 | loss=0.2602 | time=96.4s
Epoch 3/15 | loss=0.2334 | time=103.1s
Epoch 4/15 | loss=0.2186 | time=102.1s
Epoch 5/15 | loss=0.2113 | time=104.0s
Epoch 6/15 | loss=0.2048 | time=106.0s
Epoch 7/15 | loss=0.2000 | time=108.9s
Epoch 8/15 | loss=0.1967 | time=111.1s
Epoch 9/15 | loss=0.1933 | time=116.3s
Epoch 10/15 | loss=0.1898 | time=119.5s
Epoch 11/15 | loss=0.1874 | time=116.2s
Epoch 12/15 | loss=0.1848 | time=116.6s
Epoch 13/15 | loss=0.1828 | time=119.1s
Epoch 14/15 | loss=0.1813 | time=115.4s
Epoch 15/15 | loss=0.1794 | time=116.3s
{'dataset': 'KuaiRec', 'model': 'DeepFM_on', 'seed': 0, 'Recall@10': np.float64(0.017393945530019238), 'NDCG@10': np.float64(0.016172864460852185), 'Recall@20': np.float64(0.023319815138930975), 'NDCG@20': np.float64(0.020446104472388184)}

=== [KuaiRec] Training DeepFM_off (seed=0) ===
Epoch 1/15 | loss=0.3203 | time=94.8s
Epoch 2/15 | loss=0.2737 | time=95.2s
